In [ ]:
from pathlib import Path
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

CONSENSUS_DIR = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
)

DOMAIN_TIF = CONSENSUS_DIR / "PBMMAE_4domain_consensus_4km.tif"

print("Project:", PROJECT)
print("Consensus folder:", CONSENSUS_DIR)
print("Domain raster exists:", DOMAIN_TIF.exists())
print("Domain raster:", DOMAIN_TIF)

In [ ]:
with rasterio.open(DOMAIN_TIF) as src:
    domain = src.read(1)
    domain_profile = src.profile
    domain_bounds = src.bounds
    domain_crs = src.crs
    domain_transform = src.transform
    domain_nodata = src.nodata

print("Shape:", domain.shape)
print("CRS:", domain_crs)
print("Bounds:", domain_bounds)
print("NoData:", domain_nodata)

values, counts = np.unique(domain, return_counts=True)

print("\nRaster values:")
for v, n in zip(values, counts):
    print(f"{v}: {n}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# ---------------------------------------------------------
# Display final 4-domain consensus map in notebook
# ---------------------------------------------------------

# Mask formal NoData
domain_plot = domain.astype(float).copy()
domain_plot[domain_plot == -9999] = np.nan

# Discrete colors:
# -1 = tied/unresolved
#  0 = M0
#  1 = M1
#  2 = M2
#  3 = M3
colors = [
    "#D9D9D9",  # tied / unresolved
    "#4472C4",  # M0
    "#ED7D31",  # M1
    "#70AD47",  # M2
    "#A64D79",  # M3
]

cmap = ListedColormap(colors)
bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(
    domain_plot,
    cmap=cmap,
    norm=norm,
    extent=[
        domain_bounds.left,
        domain_bounds.right,
        domain_bounds.bottom,
        domain_bounds.top,
    ],
    origin="upper",
    interpolation="nearest"
)

ax.set_title("Final Four-Domain PB-MMAE Consensus", fontsize=14)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

# Colorbar
cbar = plt.colorbar(
    im,
    ax=ax,
    ticks=[-1, 0, 1, 2, 3],
    fraction=0.035,
    pad=0.03
)

cbar.ax.set_yticklabels([
    "Tied",
    "M0",
    "M1",
    "M2",
    "M3"
])

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Find candidate agreement rasters
# ---------------------------------------------------------

agreement_candidates = sorted(
    list(CONSENSUS_DIR.glob("*agreement*.tif")) +
    list(CONSENSUS_DIR.glob("*Agreement*.tif"))
)

print("Agreement raster candidates:\n")

for i, path in enumerate(agreement_candidates, start=1):
    print(f"{i}. {path.name}")

In [ ]:
# ---------------------------------------------------------
# Load consensus-agreement raster
# ---------------------------------------------------------

AGREEMENT_TIF = CONSENSUS_DIR / "PBMMAE_4domain_agreement_fraction_4km.tif"

with rasterio.open(AGREEMENT_TIF) as src:
    agreement = src.read(1).astype(float)
    agreement_bounds = src.bounds
    agreement_crs = src.crs
    agreement_nodata = src.nodata

print("Agreement raster exists:", AGREEMENT_TIF.exists())
print("Shape:", agreement.shape)
print("CRS:", agreement_crs)
print("NoData:", agreement_nodata)

# Mask formal NoData
if agreement_nodata is not None:
    agreement[agreement == agreement_nodata] = np.nan

valid_agreement = agreement[np.isfinite(agreement)]

print("\nAgreement statistics:")
print("Valid cells:", valid_agreement.size)
print("Minimum:", np.nanmin(valid_agreement))
print("Maximum:", np.nanmax(valid_agreement))
print("Mean:", np.nanmean(valid_agreement))
print("Median:", np.nanmedian(valid_agreement))

In [ ]:
from rasterio.enums import Resampling
from pyproj import Transformer

AGREEMENT_TIF = CONSENSUS_DIR / "PBMMAE_4domain_agreement_fraction_4km.tif"

with rasterio.open(AGREEMENT_TIF) as src:
    agreement_crs = src.crs
    agreement_nodata = src.nodata
    agreement_bounds_native = src.bounds

    # 4 km -> 1 km display resampling
    target_res = 1000.0
    scale_x = src.res[0] / target_res
    scale_y = src.res[1] / target_res

    new_width = int(src.width * scale_x)
    new_height = int(src.height * scale_y)

    agreement_1km = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.nearest
    )

    agreement_transform_1km = src.transform * src.transform.scale(
        src.width / new_width,
        src.height / new_height
    )

left, bottom, right, top = rasterio.transform.array_bounds(
    new_height,
    new_width,
    agreement_transform_1km
)

agreement_1km = agreement_1km.astype(float)
agreement_1km[agreement_1km == agreement_nodata] = np.nan

print("Original 4 km shape:", agreement.shape)
print("Display 1 km shape :", agreement_1km.shape)
print("CRS:", agreement_crs)
print("Display bounds:", (left, bottom, right, top))
print("Valid display cells:", np.isfinite(agreement_1km).sum())

In [ ]:
# ---------------------------------------------------------
# Load the master mask raster
# ---------------------------------------------------------
MASTER_TIF = PROJECT / "02_data" / "masks" / "Paper1_MASTER_4km.tif"

print("Master raster exists:", MASTER_TIF.exists())
print("Master raster:", MASTER_TIF)

with rasterio.open(MASTER_TIF) as src:
    master_4km = src.read(1)
    master_profile = src.profile
    master_crs = src.crs
    master_transform = src.transform
    master_nodata = src.nodata
    master_bounds = src.bounds
    master_res = src.res

print("Master shape:", master_4km.shape)
print("Master CRS:", master_crs)
print("Master bounds:", master_bounds)
print("Master resolution:", master_res)
print("Master NoData:", master_nodata)

In [ ]:
# ---------------------------------------------------------
# Build valid study-area mask from the master raster
# ---------------------------------------------------------
if master_nodata is None:
    sa_mask_4km = np.isfinite(master_4km)
else:
    sa_mask_4km = master_4km != master_nodata

print("Valid SA cells (4 km):", sa_mask_4km.sum())
print("Total cells:", sa_mask_4km.size)

In [ ]:
# =========================================================
# Rebuild 1-km agreement display using NEAREST NEIGHBOR
# display only — analytical raster remains 4 km
# =========================================================

from rasterio.enums import Resampling

with rasterio.open(AGREEMENT_TIF) as src:

    scale_x = src.res[0] / 1000.0
    scale_y = src.res[1] / 1000.0

    new_width = int(src.width * scale_x)
    new_height = int(src.height * scale_y)

    agreement_1km = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.nearest
    ).astype(float)

    agreement_transform_1km = src.transform * src.transform.scale(
        src.width / new_width,
        src.height / new_height
    )

    agreement_nodata = src.nodata

# Remove NoData correctly
agreement_1km[agreement_1km == agreement_nodata] = np.nan

left, bottom, right, top = rasterio.transform.array_bounds(
    new_height,
    new_width,
    agreement_transform_1km
)

# No additional filling/interpolation
agreement_1km_clipped = agreement_1km.copy()

print("1-km display shape:", agreement_1km_clipped.shape)
print("Valid cells:", np.isfinite(agreement_1km_clipped).sum())
print(
    "Unique agreement values:",
    np.unique(agreement_1km_clipped[np.isfinite(agreement_1km_clipped)])
)

In [ ]:
from pyproj import Transformer
from matplotlib.colors import LinearSegmentedColormap

# ---------------------------------------------------------
# Coordinate transformer: UTM 32N -> geographic
# ---------------------------------------------------------
transformer = Transformer.from_crs(master_crs, "EPSG:4326", always_xy=True)

def format_lon_dmin(x):
    lon, _ = transformer.transform(x, bottom)
    hemi = "E" if lon >= 0 else "W"
    lon = abs(lon)
    deg = int(lon)
    minutes = int(round((lon - deg) * 60))
    if minutes == 60:
        deg += 1
        minutes = 0
    return f"{deg}°{minutes:02d}′{hemi}"

def format_lat_dmin(y):
    _, lat = transformer.transform(left, y)
    hemi = "N" if lat >= 0 else "S"
    lat = abs(lat)
    deg = int(lat)
    minutes = int(round((lat - deg) * 60))
    if minutes == 60:
        deg += 1
        minutes = 0
    return f"{deg}°{minutes:02d}′{hemi}"

# ---------------------------------------------------------
# Oasis/rainbow style colormap
# ---------------------------------------------------------
oasis_rainbow = LinearSegmentedColormap.from_list(
    "oasis_rainbow",
    [
        "#313695",  # deep blue
        "#4575b4",  # blue
        "#1fa3c6",  # cyan-blue
        "#2fbf71",  # green
        "#f1e05a",  # yellow
        "#fdae61",  # orange
        "#d73027"   # red
    ],
    N=256
)

In [ ]:
# ---------------------------------------------------------
# Preview clipped agreement map
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6.8))

im = ax.imshow(
    agreement_1km_clipped,
    cmap=oasis_rainbow,
    vmin=0.0,
    vmax=1.0,
    extent=[left, right, bottom, top],
    origin="upper",
    interpolation="nearest"
)

# Optional: draw the study-area mask boundary
mask_numeric = sa_mask_1km.astype(int)
x_coords = np.linspace(left, right, mask_numeric.shape[1])
y_coords = np.linspace(top, bottom, mask_numeric.shape[0])  # note top -> bottom
ax.contour(
    x_coords,
    y_coords,
    mask_numeric,
    levels=[0.5],
    colors="black",
    linewidths=0.8
)

ax.set_title("PB-MMAE Four-Domain Consensus Agreement", fontsize=14, weight="bold")
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)

xticks = np.linspace(left, right, 6)
yticks = np.linspace(bottom, top, 5)

ax.set_xticks(xticks)
ax.set_xticklabels([format_lon_dmin(x) for x in xticks], fontsize=10)

ax.set_yticks(yticks)
ax.set_yticklabels([format_lat_dmin(y) for y in yticks], fontsize=10)

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Consensus agreement", fontsize=11)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Prepare full study-area background + valid PB-MMAE footprint
# ---------------------------------------------------------

# Full study-area rectangle from the master raster
with rasterio.open(MASTER_TIF) as src:
    master_bounds = src.bounds
    master_crs = src.crs

# Valid PB-MMAE support = any cell where agreement is not NoData
with rasterio.open(AGREEMENT_TIF) as src:
    agreement_native = src.read(1).astype(float)
    agreement_nodata = src.nodata
    agreement_transform = src.transform
    agreement_bounds = src.bounds

pb_support = agreement_native != agreement_nodata

print("PB-MMAE support cells:", pb_support.sum())
print("Expected:", 1477)

In [ ]:
# ---------------------------------------------------------
# Preview agreement within full SA extent
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6.8))

# Light-gray full study-area background
ax.set_facecolor("white")

ax.imshow(
    np.ones_like(agreement_native, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=[
        master_bounds.left,
        master_bounds.right,
        master_bounds.bottom,
        master_bounds.top
    ],
    origin="upper",
    interpolation="nearest"
)

# Mask unevaluated PB-MMAE cells
agreement_display = agreement_native.copy()
agreement_display[~pb_support] = np.nan

im = ax.imshow(
    agreement_display,
    cmap=oasis_rainbow,
    vmin=0.0,
    vmax=1.0,
    extent=[
        agreement_bounds.left,
        agreement_bounds.right,
        agreement_bounds.bottom,
        agreement_bounds.top
    ],
    origin="upper",
    interpolation="nearest"
)

ax.set_title(
    "PB-MMAE Four-Domain Consensus Agreement",
    fontsize=14,
    weight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

xticks = np.linspace(master_bounds.left, master_bounds.right, 6)
yticks = np.linspace(master_bounds.bottom, master_bounds.top, 5)

ax.set_xticks(xticks)
ax.set_xticklabels([format_lon_dmin(x) for x in xticks])

ax.set_yticks(yticks)
ax.set_yticklabels([format_lat_dmin(y) for y in yticks])

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Consensus agreement")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Preview final four-domain map using the same SA extent
# and PB-MMAE support footprint
# ---------------------------------------------------------
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Copy native domain raster
domain_display = domain.astype(float).copy()

# Formal NoData -> NaN
domain_display[domain_display == -9999] = np.nan

# Preserve only actual PB-MMAE-supported cells
domain_display[~pb_support] = np.nan

# Categorical colors
domain_cmap = ListedColormap([
    "#D9D9D9",  # tied / unresolved (-1)
    "#2F5597",  # M0
    "#ED7D31",  # M1
    "#70AD47",  # M2
    "#A64D79"   # M3
])

domain_bounds_bins = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
domain_norm = BoundaryNorm(domain_bounds_bins, domain_cmap.N)

fig, ax = plt.subplots(figsize=(10, 6.8))

# Full SA background
ax.imshow(
    np.ones_like(domain_display),
    cmap=ListedColormap(["#F7F7F7"]),
    extent=[
        master_bounds.left,
        master_bounds.right,
        master_bounds.bottom,
        master_bounds.top
    ],
    origin="upper",
    interpolation="nearest"
)

# PB-MMAE domains
ax.imshow(
    domain_display,
    cmap=domain_cmap,
    norm=domain_norm,
    extent=[
        domain_bounds.left,
        domain_bounds.right,
        domain_bounds.bottom,
        domain_bounds.top
    ],
    origin="upper",
    interpolation="nearest"
)

ax.set_title(
    "Final Four-Domain PB-MMAE Consensus",
    fontsize=14,
    weight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

xticks = np.linspace(master_bounds.left, master_bounds.right, 6)
yticks = np.linspace(master_bounds.bottom, master_bounds.top, 5)

ax.set_xticks(xticks)
ax.set_xticklabels([format_lon_dmin(x) for x in xticks])

ax.set_yticks(yticks)
ax.set_yticklabels([format_lat_dmin(y) for y in yticks])

# Clean categorical legend
legend_handles = [
    Patch(facecolor="#2F5597", label="M0"),
    Patch(facecolor="#ED7D31", label="M1"),
    Patch(facecolor="#70AD47", label="M2"),
    Patch(facecolor="#A64D79", label="M3"),
    Patch(facecolor="#D9D9D9", edgecolor="black", label="Tied")
]

ax.legend(
    handles=legend_handles,
    title="Domain",
    loc="lower right",
    frameon=True,
    fontsize=10,
    title_fontsize=10
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Panel (a): Final four-domain PB-MMAE consensus
# Final styling for Fig. 3
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# ---------------------------------------------------------
# Prepare domain raster
# ---------------------------------------------------------
domain_display = domain.astype(float).copy()

# Formal NoData -> NaN
domain_display[domain_display == -9999] = np.nan

# Keep only actual PB-MMAE-supported cells
domain_display[~pb_support] = np.nan

# ---------------------------------------------------------
# Correct class order:
# -1 = Tied
#  0 = M0
#  1 = M1
#  2 = M2
#  3 = M3
# ---------------------------------------------------------
domain_cmap = ListedColormap([
    "#9E9E9E",  # Tied
    "#2F5597",  # M0
    "#ED7D31",  # M1
    "#70AD47",  # M2
    "#A64D79"   # M3
])

domain_norm = BoundaryNorm(
    [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5],
    domain_cmap.N
)

# ---------------------------------------------------------
# Geographic bounds
# ---------------------------------------------------------
master_lon_left, master_lat_bottom = transformer.transform(
    master_bounds.left,
    master_bounds.bottom
)

master_lon_right, master_lat_top = transformer.transform(
    master_bounds.right,
    master_bounds.top
)

domain_lon_left, domain_lat_bottom = transformer.transform(
    domain_bounds.left,
    domain_bounds.bottom
)

domain_lon_right, domain_lat_top = transformer.transform(
    domain_bounds.right,
    domain_bounds.top
)

# ---------------------------------------------------------
# Coordinate ticks
# ---------------------------------------------------------
xticks = [8.0, 8.5, 9.0, 9.5, 10.0, 10.5]

yticks = [
    7 + 29/60,
    7 + 52/60,
    8 + 15/60,
    8 + 37/60,
    9.0
]

xtick_labels = [degmin_label(x, "E") for x in xticks]
ytick_labels = [degmin_label(y, "N") for y in yticks]

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(11.5, 6.8))

# Full SA background
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(
        master_lon_left,
        master_lon_right,
        master_lat_bottom,
        master_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# PB-MMAE domains
ax.imshow(
    domain_display,
    cmap=domain_cmap,
    norm=domain_norm,
    extent=(
        domain_lon_left,
        domain_lon_right,
        domain_lat_bottom,
        domain_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# ---------------------------------------------------------
# Axis labels
# ---------------------------------------------------------
ax.set_xlabel(
    "Longitude",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Latitude",
    fontsize=14,
    fontweight="bold"
)

# ---------------------------------------------------------
# Coordinates
# ---------------------------------------------------------
ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    width=1.2,
    length=5
)

# Force full SA extent
ax.set_xlim(master_lon_left, master_lon_right)
ax.set_ylim(master_lat_bottom, master_lat_top)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.97,
    "(a)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
legend_handles = [
    Patch(facecolor="#2F5597", edgecolor="black", label="M0"),
    Patch(facecolor="#ED7D31", edgecolor="black", label="M1"),
    Patch(facecolor="#70AD47", edgecolor="black", label="M2"),
    Patch(facecolor="#A64D79", edgecolor="black", label="M3"),
    Patch(facecolor="#9E9E9E", edgecolor="black", label="Tied")
]

leg = ax.legend(
    handles=legend_handles,
    title="Domain",
    loc="center left",
    bbox_to_anchor=(1.015, 0.52),
    frameon=True,
    fontsize=11,
    title_fontsize=12,
    borderaxespad=0.0,
    handlelength=1.6,
    labelspacing=0.45,
    borderpad=0.55
)

leg.get_title().set_fontweight("bold")

for txt in leg.get_texts():
    txt.set_fontweight("bold")

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

# Leave room for external legend
plt.tight_layout(rect=[0, 0, 0.88, 1])

plt.show()

In [ ]:
# =========================================================
# FIGURE 3(a) — FINAL FOUR-DOMAIN PB-MMAE CONSENSUS
# Standalone, smoothed, large-text publication version
# =========================================================

from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

CONSENSUS_DIR = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
)

DOMAIN_TIF = (
    CONSENSUS_DIR
    / "PBMMAE_4domain_consensus_4km.tif"
)

AGREEMENT_TIF = (
    CONSENSUS_DIR
    / "PBMMAE_4domain_agreement_fraction_4km.tif"
)

MASTER_TIF = (
    PROJECT
    / "02_data"
    / "masks"
    / "Paper1_MASTER_4km.tif"
)

FIG03_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "Fig03_panels"
)

FIG03_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Fixed geographic extent of the MBT–UBT grid
# ---------------------------------------------------------
lon_left = 7.999920589849341
lon_right = 10.506941457001982
lat_bottom = 7.4825594569876355
lat_top = 9.000245078529014

# ---------------------------------------------------------
# Load native rasters
# ---------------------------------------------------------
with rasterio.open(DOMAIN_TIF) as src:
    domain = src.read(1).astype(float)
    domain_nodata = src.nodata

with rasterio.open(AGREEMENT_TIF) as src:
    agreement = src.read(1).astype(float)
    agreement_nodata = src.nodata

with rasterio.open(MASTER_TIF) as src:
    master_4km = src.read(1)

if domain.shape != agreement.shape:
    raise ValueError(
        f"Domain shape {domain.shape} does not match "
        f"agreement shape {agreement.shape}."
    )

# ---------------------------------------------------------
# Prepare native domain raster
# ---------------------------------------------------------
domain_raw = domain.copy()

if domain_nodata is not None:
    domain_raw[domain_raw == domain_nodata] = np.nan

pb_support = np.isfinite(agreement)

if agreement_nodata is not None:
    pb_support &= agreement != agreement_nodata

domain_raw[~pb_support] = np.nan

# ---------------------------------------------------------
# High-resolution class-wise smoothing for DISPLAY ONLY
# ---------------------------------------------------------
UPSCALE_FACTOR = 6
SMOOTHING_PASSES = 3

def upsample_mask(mask, factor):
    return np.repeat(
        np.repeat(mask, factor, axis=0),
        factor,
        axis=1
    )

def binomial_blur_2d(array, n_passes=3):
    kernel = np.array([1, 4, 6, 4, 1], dtype=float)
    kernel /= kernel.sum()

    output = array.astype(float).copy()

    for _ in range(n_passes):

        padded_x = np.pad(
            output,
            ((0, 0), (2, 2)),
            mode="edge"
        )

        output = np.apply_along_axis(
            lambda row: np.convolve(row, kernel, mode="valid"),
            axis=1,
            arr=padded_x
        )

        padded_y = np.pad(
            output,
            ((2, 2), (0, 0)),
            mode="edge"
        )

        output = np.apply_along_axis(
            lambda col: np.convolve(col, kernel, mode="valid"),
            axis=0,
            arr=padded_y
        )

    return output


valid_support = pb_support & np.isfinite(domain_raw)

support_hr = upsample_mask(
    valid_support,
    UPSCALE_FACTOR
).astype(float)

class_scores = []

for domain_code in [0, 1, 2, 3]:

    class_mask = (domain_raw == domain_code).astype(float)

    class_mask_hr = upsample_mask(
        class_mask,
        UPSCALE_FACTOR
    )

    class_scores.append(
        binomial_blur_2d(
            class_mask_hr,
            n_passes=SMOOTHING_PASSES
        )
    )

class_scores = np.stack(class_scores, axis=0)

support_score = binomial_blur_2d(
    support_hr,
    n_passes=SMOOTHING_PASSES
)

visible_hr = support_score >= 0.50
dominant_domain_hr = np.argmax(class_scores, axis=0)

domain_display_smoothed = np.full(
    support_hr.shape,
    np.nan,
    dtype=float
)

domain_display_smoothed[visible_hr] = (
    dominant_domain_hr[visible_hr]
)

# Retain tied cells as unresolved
tied_hr = upsample_mask(
    domain_raw == -1,
    UPSCALE_FACTOR
)

domain_display_smoothed[tied_hr] = -1

# ---------------------------------------------------------
# Colors
# ---------------------------------------------------------
domain_cmap = ListedColormap([
    "#9E9E9E",  # Tied
    "#2F5597",  # M0
    "#ED7D31",  # M1
    "#70AD47",  # M2
    "#A64D79"   # M3
])

domain_norm = BoundaryNorm(
    [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5],
    domain_cmap.N
)

# ---------------------------------------------------------
# Coordinate labels
# ---------------------------------------------------------
def degmin_label(value, hemisphere):
    value_abs = abs(value)
    degree = int(np.floor(value_abs))
    minutes = int(round((value_abs - degree) * 60))

    if minutes == 60:
        degree += 1
        minutes = 0

    return f"{degree}°{minutes:02d}′{hemisphere}"


xticks = [8.0, 8.5, 9.0, 9.5, 10.0, 10.5]

yticks = [
    7 + 29 / 60,
    7 + 52 / 60,
    8 + 15 / 60,
    8 + 37 / 60,
    9.0
]

xtick_labels = [degmin_label(x, "E") for x in xticks]
ytick_labels = [degmin_label(y, "N") for y in yticks]

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(13.2, 7.2))

# Full study-area background
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(lon_left, lon_right, lat_bottom, lat_top),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Smoothed display-only domain map
ax.imshow(
    domain_display_smoothed,
    cmap=domain_cmap,
    norm=domain_norm,
    extent=(lon_left, lon_right, lat_bottom, lat_top),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Large axis labels
ax.set_xlabel("Longitude", fontsize=21, fontweight="bold")
ax.set_ylabel("Latitude", fontsize=21, fontweight="bold")

ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=17,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    width=1.4,
    length=6
)

ax.set_xlim(lon_left, lon_right)
ax.set_ylim(lat_bottom, lat_top)

# Panel label
ax.text(
    0.015,
    0.97,
    "(a)",
    transform=ax.transAxes,
    fontsize=25,
    fontweight="bold",
    va="top",
    ha="left"
)

# Large external legend
legend_handles = [
    Patch(facecolor="#2F5597", edgecolor="black", label="M0"),
    Patch(facecolor="#ED7D31", edgecolor="black", label="M1"),
    Patch(facecolor="#70AD47", edgecolor="black", label="M2"),
    Patch(facecolor="#A64D79", edgecolor="black", label="M3"),
    Patch(facecolor="#9E9E9E", edgecolor="black", label="Tied")
]

leg = ax.legend(
    handles=legend_handles,
    title="Domain",
    loc="center left",
    bbox_to_anchor=(1.015, 0.52),
    frameon=True,
    fontsize=16,
    title_fontsize=18,
    borderaxespad=0.0,
    handlelength=1.7,
    labelspacing=0.55,
    borderpad=0.65
)

leg.get_title().set_fontweight("bold")

for txt in leg.get_texts():
    txt.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout(rect=[0, 0, 0.84, 1])

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    FIG03_DIR
    / "Paper1_Fig03a_Final_4Domain_Consensus_Smoothed_LargeText_600dpi.png"
)

tif_path = (
    FIG03_DIR
    / "Paper1_Fig03a_Final_4Domain_Consensus_Smoothed_LargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# Panel (b): PB-MMAE consensus agreement
# Match panel (a) with gray full-SA background
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

# ---------------------------------------------------------
# Agreement raster
# ---------------------------------------------------------
agreement_plot = np.array(agreement_1km_clipped, dtype=float)

agreement_plot[~np.isfinite(agreement_plot)] = np.nan
agreement_plot[(agreement_plot < 0.0) | (agreement_plot > 1.0)] = np.nan

# ---------------------------------------------------------
# Oasis/rainbow-style colormap
# ---------------------------------------------------------
oasis_rainbow = LinearSegmentedColormap.from_list(
    "oasis_rainbow",
    [
        "#1f4aa8",  # blue
        "#18a1cd",  # cyan
        "#1db954",  # green
        "#f1e85b",  # yellow
        "#f4a340",  # orange
        "#e53935"   # red
    ],
    N=256
)

# ---------------------------------------------------------
# Convert master-study-area bounds to geographic coordinates
# ---------------------------------------------------------
master_lon_left, master_lat_bottom = transformer.transform(
    master_bounds.left,
    master_bounds.bottom
)

master_lon_right, master_lat_top = transformer.transform(
    master_bounds.right,
    master_bounds.top
)

# Agreement display bounds
lon_left, lat_bottom = transformer.transform(left, bottom)
lon_right, lat_top = transformer.transform(right, top)

# ---------------------------------------------------------
# Coordinate ticks
# ---------------------------------------------------------
xticks = [8.0, 8.5, 9.0, 9.5, 10.0, 10.5]
yticks = [
    7 + 29/60,
    7 + 52/60,
    8 + 15/60,
    8 + 37/60,
    9.0
]

xtick_labels = [degmin_label(x, "E") for x in xticks]
ytick_labels = [degmin_label(y, "N") for y in yticks]

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# Full SA background first
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(
        master_lon_left,
        master_lon_right,
        master_lat_bottom,
        master_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Agreement raster on top
im = ax.imshow(
    agreement_plot,
    cmap=oasis_rainbow,
    vmin=0.0,
    vmax=1.0,
    origin="upper",
    extent=(
        lon_left,
        lon_right,
        lat_bottom,
        lat_top
    ),
    interpolation="nearest",
    aspect="auto"
)

# ---------------------------------------------------------
# Axis labels
# ---------------------------------------------------------
ax.set_xlabel(
    "Longitude",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Latitude",
    fontsize=14,
    fontweight="bold"
)

# ---------------------------------------------------------
# Coordinates
# ---------------------------------------------------------
ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    width=1.2,
    length=5
)

# Force full study-area extent
ax.set_xlim(master_lon_left, master_lon_right)
ax.set_ylim(master_lat_bottom, master_lat_top)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.97,
    "(b)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

# ---------------------------------------------------------
# Colorbar
# ---------------------------------------------------------
cbar = plt.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

cbar.set_label(
    "Consensus agreement",
    fontsize=12,
    fontweight="bold"
)

cbar.ax.tick_params(labelsize=11)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# FIGURE 3(b) — PB-MMAE CONSENSUS AGREEMENT
# Standalone, smoothed, large-text publication version
# =========================================================

from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

CONSENSUS_DIR = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
)

AGREEMENT_TIF = (
    CONSENSUS_DIR
    / "PBMMAE_4domain_agreement_fraction_4km.tif"
)

MASTER_TIF = (
    PROJECT
    / "02_data"
    / "masks"
    / "Paper1_MASTER_4km.tif"
)

FIG03_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "Fig03_panels"
)

FIG03_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Fixed geographic extent of the MBT–UBT grid
# ---------------------------------------------------------
lon_left = 7.999920589849341
lon_right = 10.506941457001982
lat_bottom = 7.4825594569876355
lat_top = 9.000245078529014

# ---------------------------------------------------------
# Load master grid
# ---------------------------------------------------------
with rasterio.open(MASTER_TIF) as src:
    master_4km = src.read(1)

# ---------------------------------------------------------
# Load agreement raster and resample 4 km -> 1 km
# ---------------------------------------------------------
with rasterio.open(AGREEMENT_TIF) as src:

    target_resolution = 1000.0

    scale_x = src.res[0] / target_resolution
    scale_y = src.res[1] / target_resolution

    new_width = int(round(src.width * scale_x))
    new_height = int(round(src.height * scale_y))

    agreement_1km = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.bilinear,
        masked=True
    ).astype(float)

agreement_1km = agreement_1km.filled(np.nan)

agreement_1km[
    (agreement_1km < 0.0) | (agreement_1km > 1.0)
] = np.nan

# ---------------------------------------------------------
# Light normalized smoothing of continuous agreement
# ---------------------------------------------------------
SMOOTHING_PASSES = 3

def binomial_blur_2d(array, n_passes=3):
    kernel = np.array([1, 4, 6, 4, 1], dtype=float)
    kernel /= kernel.sum()

    output = array.astype(float).copy()

    for _ in range(n_passes):

        padded_x = np.pad(
            output,
            ((0, 0), (2, 2)),
            mode="edge"
        )

        output = np.apply_along_axis(
            lambda row: np.convolve(row, kernel, mode="valid"),
            axis=1,
            arr=padded_x
        )

        padded_y = np.pad(
            output,
            ((2, 2), (0, 0)),
            mode="edge"
        )

        output = np.apply_along_axis(
            lambda col: np.convolve(col, kernel, mode="valid"),
            axis=0,
            arr=padded_y
        )

    return output


valid_mask = np.isfinite(agreement_1km)

agreement_numerator = np.where(
    valid_mask,
    agreement_1km,
    0.0
)

agreement_weight = valid_mask.astype(float)

smooth_numerator = binomial_blur_2d(
    agreement_numerator,
    n_passes=SMOOTHING_PASSES
)

smooth_weight = binomial_blur_2d(
    agreement_weight,
    n_passes=SMOOTHING_PASSES
)

agreement_plot = np.full_like(
    agreement_1km,
    np.nan,
    dtype=float
)

valid_smooth = smooth_weight >= 0.50

agreement_plot[valid_smooth] = (
    smooth_numerator[valid_smooth]
    / smooth_weight[valid_smooth]
)

agreement_plot = np.clip(agreement_plot, 0.0, 1.0)

# ---------------------------------------------------------
# Colormap
# ---------------------------------------------------------
oasis_rainbow = LinearSegmentedColormap.from_list(
    "oasis_rainbow",
    [
        "#1f4aa8",
        "#18a1cd",
        "#1db954",
        "#f1e85b",
        "#f4a340",
        "#e53935"
    ],
    N=256
)

oasis_rainbow.set_bad((1, 1, 1, 0))

# ---------------------------------------------------------
# Coordinate labels
# ---------------------------------------------------------
def degmin_label(value, hemisphere):
    value_abs = abs(value)
    degree = int(np.floor(value_abs))
    minutes = int(round((value_abs - degree) * 60))

    if minutes == 60:
        degree += 1
        minutes = 0

    return f"{degree}°{minutes:02d}′{hemisphere}"


xticks = [8.0, 8.5, 9.0, 9.5, 10.0, 10.5]

yticks = [
    7 + 29 / 60,
    7 + 52 / 60,
    8 + 15 / 60,
    8 + 37 / 60,
    9.0
]

xtick_labels = [degmin_label(x, "E") for x in xticks]
ytick_labels = [degmin_label(y, "N") for y in yticks]

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(12.0, 7.2))

# Full study-area background
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(lon_left, lon_right, lat_bottom, lat_top),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Smoothed agreement field
im = ax.imshow(
    agreement_plot,
    cmap=oasis_rainbow,
    vmin=0.0,
    vmax=1.0,
    origin="upper",
    extent=(lon_left, lon_right, lat_bottom, lat_top),
    interpolation="bilinear",
    aspect="auto"
)

# Axis labels
ax.set_xlabel("Longitude", fontsize=21, fontweight="bold")
ax.set_ylabel("Latitude", fontsize=21, fontweight="bold")

ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=17,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    width=1.4,
    length=6
)

ax.set_xlim(lon_left, lon_right)
ax.set_ylim(lat_bottom, lat_top)

# Panel label
ax.text(
    0.015,
    0.97,
    "(b)",
    transform=ax.transAxes,
    fontsize=25,
    fontweight="bold",
    va="top",
    ha="left"
)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

# Colorbar
cbar = plt.colorbar(
    im,
    ax=ax,
    fraction=0.042,
    pad=0.035
)

cbar.set_label(
    "Consensus agreement",
    fontsize=19,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=16,
    width=1.2,
    length=5
)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    FIG03_DIR
    / "Paper1_Fig03b_Consensus_Agreement_Smoothed_LargeText_600dpi.png"
)

tif_path = (
    FIG03_DIR
    / "Paper1_Fig03b_Consensus_Agreement_Smoothed_LargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# Panel (c): Hierarchical stability of meta-domain solutions
# Final styling
# =========================================================

import numpy as np
import matplotlib.pyplot as plt

# Frozen hierarchy metrics
meta_n = np.array([2, 3, 4, 5, 6, 7, 8])

mean_agreement = np.array([
    0.890,
    0.853,
    0.823,
    0.763,
    0.737,
    0.720,
    0.694
])

full_agreement = np.array([
    0.677,
    0.581,
    0.517,
    0.415,
    0.355,
    0.339,
    0.309
])

tie_fraction = np.array([
    0.018,
    0.036,
    0.064,
    0.154,
    0.172,
    0.206,
    0.249
])

fig, ax = plt.subplots(figsize=(8.8, 5.5))

# ---------------------------------------------------------
# Stability curves
# ---------------------------------------------------------
ax.plot(
    meta_n,
    mean_agreement,
    marker="o",
    linewidth=2.4,
    markersize=7,
    label="Mean agreement"
)

ax.plot(
    meta_n,
    full_agreement,
    marker="s",
    linewidth=2.4,
    markersize=7,
    label="Full agreement"
)

ax.plot(
    meta_n,
    tie_fraction,
    marker="^",
    linewidth=2.4,
    markersize=7,
    label="Tie fraction"
)

# ---------------------------------------------------------
# Retained four-domain solution
# ---------------------------------------------------------
ax.axvline(
    4,
    color="#444444",
    linestyle="--",
    linewidth=1.7
)

ax.text(
    4.10,
    0.94,
    "Retained\n4-domain solution",
    fontsize=11,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.97,
    "(c)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    "Number of meta-domains",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Stability metric",
    fontsize=14,
    fontweight="bold"
)

ax.set_xticks(meta_n)

ax.set_ylim(0, 1.0)

ax.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
leg = ax.legend(
    loc="upper right",
    frameon=True,
    fontsize=10.5,
    borderpad=0.55,
    labelspacing=0.45
)

for txt in leg.get_texts():
    txt.set_fontweight("bold")

# ---------------------------------------------------------
# Light horizontal grid only
# ---------------------------------------------------------
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# FIGURE 3(c) — HIERARCHICAL STABILITY OF META-DOMAIN CUTS
# Large-text publication version
# =========================================================

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Frozen hierarchy metrics
# Global metrics across all mapped cells and meta-domains
# ---------------------------------------------------------
meta_n = np.array([2, 3, 4, 5, 6, 7, 8])

mean_agreement = np.array([
    0.890,
    0.853,
    0.823,
    0.763,
    0.737,
    0.720,
    0.694
])

full_agreement = np.array([
    0.677,
    0.581,
    0.517,
    0.415,
    0.355,
    0.339,
    0.309
])

tie_fraction = np.array([
    0.018,
    0.036,
    0.064,
    0.154,
    0.172,
    0.206,
    0.249
])

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(12.0, 7.2))

# Stability curves
ax.plot(
    meta_n,
    mean_agreement,
    marker="o",
    linewidth=3.0,
    markersize=11,
    label="Mean agreement",
    zorder=3
)

ax.plot(
    meta_n,
    full_agreement,
    marker="s",
    linewidth=3.0,
    markersize=11,
    label="Full agreement",
    zorder=3
)

ax.plot(
    meta_n,
    tie_fraction,
    marker="^",
    linewidth=3.0,
    markersize=11,
    label="Tie fraction",
    zorder=3
)

# Retained four-domain cut
ax.axvline(
    4,
    color="#444444",
    linestyle="--",
    linewidth=2.2,
    zorder=2
)

ax.text(
    4.12,
    0.965,
    "Retained\n4-domain solution",
    fontsize=17,
    fontweight="bold",
    va="top",
    ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.88,
        pad=1.0
    ),
    zorder=5
)

# Panel label
ax.text(
    0.015,
    0.97,
    "(c)",
    transform=ax.transAxes,
    fontsize=25,
    fontweight="bold",
    va="top",
    ha="left"
)

# Axes
ax.set_xlabel(
    "Number of meta-domains",
    fontsize=21,
    fontweight="bold"
)

ax.set_ylabel(
    "Stability metric",
    fontsize=21,
    fontweight="bold"
)

ax.set_xticks(meta_n)
ax.set_xlim(1.7, 8.35)
ax.set_ylim(0, 1.0)

ax.tick_params(
    axis="both",
    labelsize=17,
    width=1.4,
    length=6
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

# Legend
leg = ax.legend(
    loc="upper right",
    frameon=True,
    fontsize=16,
    borderpad=0.60,
    labelspacing=0.45
)

for txt in leg.get_texts():
    txt.set_fontweight("bold")

# Grid and frame
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
FIG03_DIR = (
    Path(r"C:\project 3\Paper1_SSL")
    / "03_outputs"
    / "figures"
    / "Fig03_panels"
)

FIG03_DIR.mkdir(parents=True, exist_ok=True)

png_path = (
    FIG03_DIR
    / "Paper1_Fig03c_Hierarchical_Stability_LargeText_600dpi.png"
)

tif_path = (
    FIG03_DIR
    / "Paper1_Fig03c_Hierarchical_Stability_LargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# EXPORT FIG. 3 PANELS INDIVIDUALLY
# 600 dpi for final PowerPoint assembly
# =========================================================

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ---------------------------------------------------------
# Output folder
# ---------------------------------------------------------
FIG03_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "Fig03_panels"
)

FIG03_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:")
print(FIG03_DIR)


# =========================================================
# PANEL (a) — FINAL FOUR-DOMAIN CONSENSUS
# =========================================================

fig, ax = plt.subplots(figsize=(8.5, 6.0))

# Full SA background
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(
        master_lon_left,
        master_lon_right,
        master_lat_bottom,
        master_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Domain raster
ax.imshow(
    domain_display,
    cmap=domain_cmap,
    norm=domain_norm,
    extent=(
        domain_lon_left,
        domain_lon_right,
        domain_lat_bottom,
        domain_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

ax.set_xlim(master_lon_left, master_lon_right)
ax.set_ylim(master_lat_bottom, master_lat_top)

ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")

ax.text(
    0.015, 0.97, "(a)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
)

legend_handles = [
    Patch(facecolor="#2F5597", edgecolor="black", label="M0"),
    Patch(facecolor="#ED7D31", edgecolor="black", label="M1"),
    Patch(facecolor="#70AD47", edgecolor="black", label="M2"),
    Patch(facecolor="#A64D79", edgecolor="black", label="M3"),
    Patch(facecolor="#9E9E9E", edgecolor="black", label="Tied")
]

leg = ax.legend(
    handles=legend_handles,
    title="Domain",
    loc="center left",
    bbox_to_anchor=(1.02, 0.52),
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

leg.get_title().set_fontweight("bold")

for txt in leg.get_texts():
    txt.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

fig.savefig(
    FIG03_DIR / "Paper1_Fig03a_Final_4Domain_Consensus_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIG03_DIR / "Paper1_Fig03a_Final_4Domain_Consensus_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


# =========================================================
# PANEL (b) — CONSENSUS AGREEMENT
# =========================================================

fig, ax = plt.subplots(figsize=(8.5, 6.0))

# Full SA background
ax.imshow(
    np.ones_like(master_4km, dtype=float),
    cmap=ListedColormap(["#F2F2F2"]),
    extent=(
        master_lon_left,
        master_lon_right,
        master_lat_bottom,
        master_lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

# Agreement raster
im = ax.imshow(
    agreement_plot,
    cmap=oasis_rainbow,
    vmin=0.0,
    vmax=1.0,
    extent=(
        lon_left,
        lon_right,
        lat_bottom,
        lat_top
    ),
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

ax.set_xlim(master_lon_left, master_lon_right)
ax.set_ylim(master_lat_bottom, master_lat_top)

ax.set_xticks(xticks)
ax.set_xticklabels(
    xtick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    ytick_labels,
    fontsize=12,
    fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")

ax.text(
    0.015, 0.97, "(b)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

cbar.set_label(
    "Consensus agreement",
    fontsize=12,
    fontweight="bold"
)

cbar.ax.tick_params(labelsize=11)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

fig.savefig(
    FIG03_DIR / "Paper1_Fig03b_Consensus_Agreement_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIG03_DIR / "Paper1_Fig03b_Consensus_Agreement_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


# =========================================================
# PANEL (c) — HIERARCHICAL STABILITY
# =========================================================

fig, ax = plt.subplots(figsize=(8.5, 5.5))

ax.plot(
    meta_n,
    mean_agreement,
    marker="o",
    linewidth=2.4,
    markersize=7,
    label="Mean agreement"
)

ax.plot(
    meta_n,
    full_agreement,
    marker="s",
    linewidth=2.4,
    markersize=7,
    label="Full agreement"
)

ax.plot(
    meta_n,
    tie_fraction,
    marker="^",
    linewidth=2.4,
    markersize=7,
    label="Tie fraction"
)

ax.axvline(
    4,
    color="#444444",
    linestyle="--",
    linewidth=1.7
)

ax.text(
    4.10,
    0.94,
    "Retained 4-domain solution",
    fontsize=11,
    fontweight="bold",
    va="top"
)

ax.text(
    0.015, 0.97, "(c)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
)

ax.set_xlabel(
    "Number of meta-domains",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Stability metric",
    fontsize=14,
    fontweight="bold"
)

ax.set_xticks(meta_n)
ax.set_ylim(0, 1.0)

ax.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

leg = ax.legend(
    loc="upper right",
    frameon=True,
    fontsize=11
)

for txt in leg.get_texts():
    txt.set_fontweight("bold")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

fig.savefig(
    FIG03_DIR / "Paper1_Fig03c_Hierarchical_Stability_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIG03_DIR / "Paper1_Fig03c_Hierarchical_Stability_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

# Vector version is particularly useful for the graph
fig.savefig(
    FIG03_DIR / "Paper1_Fig03c_Hierarchical_Stability.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


# =========================================================
# CONFIRM OUTPUTS
# =========================================================

print("\nSaved Fig. 3 panels:\n")

for p in sorted(FIG03_DIR.iterdir()):
    print(p.name)